In [3]:
# ============================================================
# Google Colab setup
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

REPO_NAME = "CI_tutorial"
REPO_URL = "https://github.com/tiksato/CI_tutorial.git"
REPO_DIR = Path("/content") / REPO_NAME

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

    os.chdir(REPO_DIR)

    # Install dependencies from pyproject.toml
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", "."],
        check=True
    )

    # Then move to notebooks directory if desired
    os.chdir(REPO_DIR / "notebooks")

# Add parent of CI_tutorial to sys.path
# This allows imports like:
# from CI_tutorial.modules... import ...
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    REPO_DIR = NOTEBOOK_DIR.parent
else:
    REPO_DIR = NOTEBOOK_DIR

PARENT_DIR = REPO_DIR.parent

if str(PARENT_DIR) not in sys.path:
    sys.path.insert(0, str(PARENT_DIR))

print("Current directory:", Path.cwd())
print("Repository:", REPO_DIR)
print("Python path added:", PARENT_DIR)
print("Setup completed.")

Current directory: /content/CI_tutorial/notebooks
Repository: /content/CI_tutorial
Python path added: /content
Setup completed.


In [4]:
import time
import math
import numpy as np
from pyscf import gto
from CI_tutorial.modules.misc_utils.pyscf_tools import get_integrals_rhf
from CI_tutorial.modules.misc_utils.pyscf_tools import davidson_restricted_fullci_mask as davidson_pyscf
from CI_tutorial.modules.misc_utils.matrix_utilities import davidson
from CI_tutorial.modules.ormas_tools.ormas2 import ORMAS2

# Maximum CI dimension for memory safety
max_dim_ormas     =  200000000 # 0.2 billion
max_dim_fci_pyscf = 1000000000 # 1 billion

In [5]:
# We consider an Ar atom, with (9,9) electrons in total,
# with 6-31G** basis, which amounts to 18 basis functions.
mol = gto.M(
    atom='Ar 0 0 0',
    basis='6-31G**',
    symmetry=False,
    verbose=4
)
# Atomic orbitals are characterized by
# 1s, 2s, 2p, 3s, 3p, 4s, 4p, and 3d orbitals in
# acsending order of orbital energy:
#  mo_energy =
#[-118.59460559  -12.31810372   -9.5684429    -9.5684429    -9.5684429
#   -1.27474354   -0.58891803   -0.58891803   -0.58891803    0.62666152
#    0.72602205    0.72602205    0.72602205    1.21762909    1.21762909
#    1.21762909    1.21762909    1.21762909]

System: uname_result(system='Linux', node='dc268ca63fbe', release='6.6.122+', version='#1 SMP Thu Apr 30 18:17:14 UTC 2026', machine='x86_64')  Threads 2
Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.16.0
Date: Mon Jun 15 03:36:10 2026
PySCF version 2.13.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 1
[INPUT] num. electrons = 18
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 Ar     0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr   0.0

nuclear repulsion = 0
number of shells = 8
number of NR pGTOs = 51
number of NR cGTOs = 18
basis = 6-31G**
ecp = {}
CPU time:         3.94


In [6]:
# Freeze 1s, 2s, 2p to prepare an effective
# (13o, 8e) active space problem.
n_core = 5        # core orbitals: 1s,2s,2p ((5,5) electrons)
n_act = 13        # active orbitals: 3s,3p, and 4s,4p,3d
n_virt = 0        # no virtual orbitals
n_elec = (4,4)    # active electrons, each spin
n_elec_total = 8  # active electrons

In [7]:
# CASCI(3s, 3p, 4s, 4p, 3d)
occ_info_spin = [
    [
        {'n_orb':13, 'min':-1, 'max':-1}, # 3s,3p,4s,4p,3d: free occupation
    ],
    [
        {'n_orb':13, 'min':-1, 'max':-1}, # 3s,3p,4s,4p,3d: free occupation
    ]
]
occ_info_total = [
        {'n_orb':26, 'min':-1, 'max':-1}, # 3s,3p,4s,4p,3d: free occupation
]

In [9]:
########################################################################
# CI instance generation
print("\n")
print("#"*72)
print("[Step 1] Constructing CI object...")
t0 = time.time()
myCI = ORMAS2(n_elec,
              occ_info_spin,
              occ_info_total,
              num_threads = 2,
              verbose = 1)
print(f" ... Done: {time.time()-t0:.2f} sec.")
print(f"CI dimension: {myCI.total_dim}")
print("String distribution over occupation groups:")
print(myCI.mat_num_str)



########################################################################
[Step 1] Constructing CI object...
ORMAS1: num_threads =  2
ORMAS1: Allowed distributions 0.000 seconds
ORMAS1: generate strings 2.394 seconds
ORMAS1: build_string_to_index 1.856 seconds
ORMAS1: _generate_trans1 8.475 seconds
ORMAS1: _generate_trans2 2.729 seconds
ORMAS1: num_threads =  2
ORMAS1: Allowed distributions 0.000 seconds
ORMAS1: generate strings 0.005 seconds
ORMAS1: build_string_to_index 0.002 seconds
ORMAS1: _generate_trans1 0.009 seconds
ORMAS1: _generate_trans2 0.070 seconds
ORMAS1 objects construction 16.665 seconds
generate_distributions 0.000 seconds
generate_occupation_boundaries 0.000 seconds
generate_determinant_offsets 3.003 seconds
generate_transpose_map 0.666 seconds
generate_trans1_det_boundaries 0.703 seconds
generate_trans2_det_boundaries 0.031 seconds
generate_trans1_for_pq_valid 1.674 seconds
 ... Done: 22.78 sec.
CI dimension: 511225
String distribution over occupation groups:
[[511

In [10]:
########################################################################
# Hamiltonian matrix elements generation
print("\n")
print("#"*72)
print("[Step 2] Running RHF to obtain MO integrals...")
t0 = time.time()
e_core, h1eff, h2_phys = get_integrals_rhf(mol, n_core, n_act, n_elec_total)
print(f" ... Done: {time.time()-t0:.2f} sec.")
print(f"Core energy: {e_core}")



########################################################################
[Step 2] Running RHF to obtain MO integrals...
RHF Total Energy: -526.7721510920 Hartree (Time: 0.10 sec)
Integral transformation completed (Time: 0.00 sec)
 ... Done: 0.10 sec.
Core energy: -505.90328302520606


In [11]:
########################################################################
# Direct-CI Davidson diagonalization
print("\n")
print("#"*72)
if myCI.total_dim < max_dim_ormas:
    print("[Step 3] ORMAS Davidson diagonalization...")
    t0 = time.time()
    def my_get_sigma(x):
        return myCI.h_prod(h1eff, h2_phys, x).real
        #return myCI.h_prod_force_symmetric(h1eff, h2_phys, x).real
        return get_sigma(x).real

    hdiag = myCI.calc_hdiag(h1eff, h2_phys).real
    def my_precond(dx, e, x0):
        denom = hdiag - e
        denom[abs(denom) < 1e-8] = 1e-8  # avoid zero division
        return dx / denom

    E1, U1 = davidson(myCI.total_dim,
                      my_get_sigma,
                      my_precond,
                      verbose=5)
    print(f" ... Done: {time.time()-t0:.2f} sec.")
    print(f"ORMAS CI energy: {E1 + e_core}")
else:
    print("[Step 3] myCI.total_dim too large. Skip ORMAS diagonalization.")



########################################################################
[Step 3] ORMAS Davidson diagonalization...
verbose = 5
davidson 0 1  |r|= 0.686  e= [-20.86886807]  max|de|= -20.9  lindep=    1
davidson 1 2  |r|= 0.191  e= [-21.01055498]  max|de|= -0.142  lindep=    1
davidson 2 3  |r|= 0.041  e= [-21.01757095]  max|de|= -0.00702  lindep= 0.999
davidson 3 4  |r|= 0.00825  e= [-21.0178641]  max|de|= -0.000293  lindep= 0.981
davidson 4 5  |r|= 0.00167  e= [-21.0178767]  max|de|= -1.26e-05  lindep= 0.957
davidson 5 6  |r|= 0.000379  e= [-21.01787719]  max|de|= -4.92e-07  lindep= 0.93
davidson 6 7  |r|= 0.000101  e= [-21.01787722]  max|de|= -2.87e-08  lindep= 0.955
davidson 7 8  |r|= 2.7e-05  e= [-21.01787722]  max|de|= -1.93e-09  lindep= 0.912
davidson 8 9  |r|= 6.03e-06  e= [-21.01787722]  max|de|= -1.26e-10  lindep= 0.959
davidson 9 10  |r|= 1.45e-06  e= [-21.01787722]  max|de|= -7.03e-12  lindep= 0.892
root 0 converged  |r|= 2.76e-07  e= -21.01787722043162  max|de|= -4.23e-13